# Notebook 03 — Experiment 2 orchestration comparison

Canonical analysis entry point for **Experiment 2: AaT vs Plan-Execute (and optional Verified PE)**.

Scope:

| Cell | Label | Orchestration | Directory |
|---|---|---|---|
| B | AaT MCP-baseline | `agent_as_tool` | `benchmarks/cell_B_mcp_baseline/` |
| Y | PE | `plan_execute` | `benchmarks/cell_Y_plan_execute/` |
| Z | Verified PE | `verified_pe` | `benchmarks/cell_Z_hybrid/` |

Cell B is **shared** with Experiment 1 — it's the AaT MCP-baseline. That's why it shows up in both Notebook 02 (where it's the transport-overhead anchor) and Notebook 03 (where it's the AaT orchestration baseline). The `CONTRIBUTING_EXPERIMENTS="exp1_mcp_overhead,exp2_orchestration"` tag on Cell B's config makes this explicit.

Cell Z still writes into the legacy benchmark directory `benchmarks/cell_Z_hybrid/`, but the method name is now **Verified PE** and the canonical Experiment 2 config lives at `configs/experiment2/exp2_cell_Z_verified_pe_mcp_baseline.env`.

Runner output shape (per-scenario JSON written by `run_experiment.sh` + AOB / repo-local runners):

- `success` — bool, true if no failed_steps (may be `null` on legacy pre-normalization artifacts; treated as unknown/NaN)
- `failed_steps` — list of step indices (or descriptors) that failed
- `history` — list of step entries; each carries `success`, `error`, `response` (may include nested `error` payload)
- `plan` — the initial plan (for PE / Verified PE)
- `answer` — final answer string
- per-file latency in `latencies.jsonl` under the run dir

Normalized-runner contract (from `scripts/orchestration_utils.py` on `codex-fnd/issue-23-24-verified-pe-self-ask`): when a tool call returns an error payload, the runner promotes the error to `step["error"]` and sets `step["success"] = False` atomically. So a single failed step should only contribute **one** tool-error count — not two.

This notebook is now **staged**, not all-or-nothing:

- always write the availability snapshot
- if Y and Z are both present, write the PE-family follow-on comparison
- if B and Y are present, write the core Experiment 2 comparison
- if raw baseline + Self-Ask pairs exist for Y and/or Z, write the Self-Ask ablation inventory and comparison outputs
- include Z in the core table/figure when it is present too


In [1]:
import json
import re
import platform
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "benchmarks").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from notebook path")

REPO_ROOT = find_repo_root(Path.cwd())
BENCHMARKS_DIR = REPO_ROOT / "benchmarks"
RESULTS_METRICS_DIR = REPO_ROOT / "results" / "metrics"
RESULTS_FIGURES_DIR = REPO_ROOT / "results" / "figures"
RESULTS_METRICS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Declare every output path up front so degraded/zero-cell branches can replace
# (not silently leave) stale prior-run artifacts. See PR #123 v2 reviewer
# guidance: a fresh availability CSV alongside a stale comparison figure is a
# correctness-class regression, not a UX nit.
AVAILABILITY_CSV = RESULTS_METRICS_DIR / "notebook03_cell_availability.preflight.csv"
ORCHESTRATION_CSV = RESULTS_METRICS_DIR / "notebook03_orchestration_comparison.csv"
PE_FAMILY_CSV = RESULTS_METRICS_DIR / "notebook03_pe_family_follow_on.csv"
FAILURE_BREAKDOWN_CSV = RESULTS_METRICS_DIR / "notebook03_failure_breakdown.csv"
SELF_ASK_INVENTORY_CSV = RESULTS_METRICS_DIR / "notebook03_self_ask_run_inventory.preflight.csv"
SELF_ASK_ABLATION_CSV = RESULTS_METRICS_DIR / "notebook03_self_ask_ablation.csv"
ORCHESTRATION_FIGURE_PNG = RESULTS_FIGURES_DIR / "notebook03_orchestration_comparison.png"
PE_FAMILY_FIGURE_PNG = RESULTS_FIGURES_DIR / "notebook03_pe_family_follow_on.png"
SELF_ASK_FIGURE_PNG = RESULTS_FIGURES_DIR / "notebook03_self_ask_ablation.png"

ORCHESTRATION_CSV_COLUMNS = [
    "cell", "label", "mode", "scenarios", "success_rate",
    "mean_failed_steps", "mean_history_length", "mean_tool_errors", "mean_replans",
]
FAILURE_BREAKDOWN_COLUMNS = [
    "cell", "label", "scenarios", "any_failure", "recovered", "recovery_rate",
]
SELF_ASK_ABLATION_COLUMNS = [
    "cell", "comparison_label", "variant_key", "variant_label",
    "run_id", "run_name", "scenarios", "success_rate",
    "mean_failed_steps", "mean_history_length", "mean_tool_errors", "mean_replans",
]


def _write_deferred_csv(path: Path, columns: list[str], note: str) -> None:
    """Replace any stale CSV at `path` with a schema-only frame plus a note row."""
    columns_with_note = [*columns, "note"] if "note" not in columns else columns
    note_row = {col: "" for col in columns_with_note}
    note_row["note"] = note
    pd.DataFrame([note_row], columns=columns_with_note).to_csv(path, index=False)


def _write_deferred_figure(path: Path, title: str, message: str) -> None:
    """Replace any stale PNG at `path` with a placeholder figure carrying `message`."""
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.5, message, ha="center", va="center", fontsize=11, color="#333333", wrap=True)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor("#cccccc")
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)


print(f"Repo root:   {REPO_ROOT}")
print(f"Python:      {platform.python_version()}")
print(f"pandas:      {pd.__version__}")
print(f"numpy:       {np.__version__}")
print(f"matplotlib:  {matplotlib.__version__}")

Repo root:   /Users/wax/coding/hpml-assetopsbench-smart-grid-mcp/.codex/worktrees/codex-fnd-exp-32-34-experiment2
Python:      3.12.8
pandas:      3.0.0
numpy:       2.2.6
matplotlib:  3.10.8


## Cells scanned

In [3]:
CELL_SPECS = [
    {"cell": "B", "label": "AaT (MCP baseline)", "path": BENCHMARKS_DIR / "cell_B_mcp_baseline"},
    {"cell": "Y", "label": "Plan-Execute",        "path": BENCHMARKS_DIR / "cell_Y_plan_execute"},
    {"cell": "Z", "label": "Verified PE", "path": BENCHMARKS_DIR / "cell_Z_hybrid"},
]

SUMMARY_FIELDS = [
    "experiment_family", "experiment_cell", "orchestration_mode", "mcp_mode",
    "scenario_set_name", "scenario_set_hash", "model_id",
    "run_status", "scenarios_attempted", "scenarios_completed",
    "success_rate", "failure_count",
    "wall_clock_seconds_total",
    "latency_seconds_mean", "latency_seconds_p50", "latency_seconds_p95",
    "tool_call_count_total", "tool_call_count_mean", "tool_error_count",
    "mcp_latency_seconds_mean", "mcp_latency_seconds_p95",
    "tool_latency_seconds_mean",
    "wandb_run_url",
    "judge_score_mean", "judge_score_p50", "judge_score_p95",
    "judge_pass_rate",
]

def _load_json(path: Path):
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        return None

def _load_jsonl(path: Path) -> tuple[list[dict], list[dict]]:
    """Read a JSONL file. Returns (rows, parse_errors).

    Each parse_error is `{"path": str(path), "line_number": int, "raw": str, "reason": str}`.
    Caller decides loudness: cheap counts ignore errors, contract validators
    surface them as explicit failing checks so corrupt artifacts cannot hide.
    """
    rows: list[dict] = []
    errors: list[dict] = []
    if path is None or not path.exists():
        return rows, errors
    with path.open() as handle:
        for lineno, line in enumerate(handle, start=1):
            stripped = line.strip()
            if not stripped:
                continue
            try:
                rows.append(json.loads(stripped))
            except json.JSONDecodeError as exc:
                errors.append({
                    "path": str(path),
                    "line_number": lineno,
                    "raw": stripped[:200],
                    "reason": str(exc),
                })
    return rows, errors


def _count_jsonl_rows(path: Path) -> int:
    rows, _ = _load_jsonl(path)
    return len(rows)

def _parse_iso_ts(ts: str | None):
    """Return a timezone-aware datetime from an ISO-8601 string, or None.

    Handles the `Z` suffix by converting to `+00:00` before parsing. If the
    string is missing a timezone offset entirely, it is treated as naive and
    returned as-is — downstream code must not mix naive and aware datetimes.
    """
    if not ts:
        return None
    try:
        normalized = ts.replace("Z", "+00:00") if ts.endswith("Z") else ts
        return datetime.fromisoformat(normalized)
    except (TypeError, ValueError):
        return None

def _latest_run_dir(raw_dir: Path) -> Path | None:
    """Pick the latest run dir by parsed `meta.json.started_at`; fall back to mtime.

    Raw string comparison on ISO-8601 is not chronological across mixed offsets
    (e.g. `2026-04-21T00:00:00-04:00` > `2026-04-21T00:00:00Z` lexicographically
    but NOT chronologically), so we parse into timezone-aware datetimes first.
    Entries with unparseable or missing timestamps fall back to directory mtime.
    """
    if not raw_dir.exists():
        return None
    candidates = [p for p in raw_dir.iterdir() if p.is_dir()]
    if not candidates:
        return None

    with_ts, without_ts = [], []
    for p in candidates:
        meta_path = p / "meta.json"
        parsed = None
        if meta_path.exists():
            meta = _load_json(meta_path)
            if meta:
                parsed = _parse_iso_ts(meta.get("started_at") or meta.get("finished_at"))
        if parsed is not None:
            with_ts.append((parsed, p))
        else:
            without_ts.append((p.stat().st_mtime, p))

    if with_ts:
        return max(with_ts, key=lambda pair: pair[0])[1]
    return max(without_ts, key=lambda pair: pair[0])[1]

def _scenario_jsons(run_dir: Path):
    if not run_dir.exists():
        return []
    return sorted(
        p for p in run_dir.glob("*.json")
        if p.name not in {"meta.json", "summary.json", "config.json"}
    )

def scan_cell(cell_spec: dict) -> dict:
    cell_dir = cell_spec["path"]
    raw_dir = cell_dir / "raw"
    run_dirs = sorted([p for p in raw_dir.iterdir() if p.is_dir()]) if raw_dir.exists() else []
    latest_run = _latest_run_dir(raw_dir)

    summary = _load_json(cell_dir / "summary.json")
    config = _load_json(cell_dir / "config.json")
    latencies_path = latest_run / "latencies.jsonl" if latest_run else None
    scenario_files = _scenario_jsons(latest_run) if latest_run else []

    row = {
        "cell": cell_spec["cell"],
        "label": cell_spec["label"],
        "cell_dir": str(cell_dir.relative_to(REPO_ROOT)),
        "config_present": (cell_dir / "config.json").exists(),
        "summary_present": (cell_dir / "summary.json").exists(),
        "raw_run_count": len(run_dirs),
        "latest_run_id": latest_run.name if latest_run else "",
        "latency_rows": _count_jsonl_rows(latencies_path) if latencies_path else 0,
        "scenario_json_count": len(scenario_files),
    }
    for field in SUMMARY_FIELDS:
        row[field] = summary.get(field) if summary else None
    return row

availability_df = pd.DataFrame(scan_cell(spec) for spec in CELL_SPECS)
availability_df

,cell,label,cell_dir,config_present,summary_present,raw_run_count,latest_run_id,latency_rows,scenario_json_count,experiment_family,...,tool_call_count_mean,tool_error_count,mcp_latency_seconds_mean,mcp_latency_seconds_p95,tool_latency_seconds_mean,wandb_run_url,judge_score_mean,judge_score_p50,judge_score_p95,judge_pass_rate
0,B,AaT (MCP baseline),benchmarks/cell_B_mcp_baseline,True,True,1,8979314_aat_mcp_baseline,6,6,exp1_mcp_overhead,...,3.5,0,None,None,None,https://wandb.ai/assetopsbench-smartgrid/asset...,None,None,None,None
1,Y,Plan-Execute,benchmarks/cell_Y_plan_execute,True,True,3,local-20260413-003914_pe_mcp_baseline_watsonx_...,1,1,exp2_orchestration,...,9.5,0,None,None,None,https://wandb.ai/assetopsbench-smartgrid/asset...,None,None,None,None
2,Z,Verified PE,benchmarks/cell_Z_hybrid,True,True,0,,0,0,exp2_orchestration,...,10.5,0,None,None,None,https://wandb.ai/assetopsbench-smartgrid/asset...,None,None,None,None


In [4]:
availability_df.to_csv(AVAILABILITY_CSV, index=False)
print(f"Wrote {AVAILABILITY_CSV.relative_to(REPO_ROOT)}")

display_cols = [
    "cell", "label",
    "config_present", "summary_present", "raw_run_count",
    "latest_run_id", "latency_rows", "scenario_json_count",
    "orchestration_mode", "mcp_mode",
    "success_rate", "latency_seconds_p50", "latency_seconds_p95",
    "tool_call_count_mean", "tool_error_count",
    "judge_score_mean", "judge_pass_rate",
]
display(availability_df[display_cols])

Wrote results/metrics/notebook03_cell_availability.preflight.csv


,cell,label,config_present,summary_present,raw_run_count,latest_run_id,latency_rows,scenario_json_count,orchestration_mode,mcp_mode,success_rate,latency_seconds_p50,latency_seconds_p95,tool_call_count_mean,tool_error_count,judge_score_mean,judge_pass_rate
0,B,AaT (MCP baseline),True,True,1,8979314_aat_mcp_baseline,6,6,agent_as_tool,baseline,1.0,12.909865,16.654413,3.5,0,None,None
1,Y,Plan-Execute,True,True,3,local-20260413-003914_pe_mcp_baseline_watsonx_...,1,1,plan_execute,baseline,1.0,34.647748,99.566396,9.5,0,None,None
2,Z,Verified PE,True,True,0,,0,0,verified_pe,baseline,1.0,47.544014,139.637523,10.5,0,None,None


## Readiness gate

Notebook 03 now has three useful stages:

1. **Preflight only** — fewer than two relevant cells are ready.
2. **PE-family follow-on mode** — Y and Z are both ready, so we can analyze PE vs Verified PE and their ablation ladder.
3. **Core Experiment 2 mode** — B and Y are both ready, so we can analyze the honest AaT vs PE comparison. If Z is also ready, include it as the optional third-method lane.

Readiness per cell still means:

- a committed `config.json`
- a committed `summary.json`
- at least one raw run directory with ≥1 per-scenario JSON

Note: the earlier Verified PE entry in `docs/validation_log.md` (`8851966_verified_pe_mcp_baseline_smoke`) is useful debugging history, but the clean benchmark-ready reference point is the later `2 / 2` smoke success at `8857843_verified_pe_mcp_baseline_smoke`. Use the newest clean validation entry as the baseline when this notebook graduates from preflight mode.

Legacy artifacts (pre-normalization) that lack canonical `scenario.id` and `success` fields are **excluded** from the scenario-level analysis below, not silently coerced — see the `scenario_status` column in the per-scenario extract.


In [5]:
availability_df["analysis_ready"] = (
    availability_df["config_present"]
    & availability_df["summary_present"]
    & (availability_df["raw_run_count"] > 0)
    & (availability_df["scenario_json_count"] > 0)
)

ready_cells = availability_df.loc[availability_df["analysis_ready"], "cell"].tolist()
missing_cells = availability_df.loc[~availability_df["analysis_ready"], "cell"].tolist()
ready_set = set(ready_cells)

core_ready = {"B", "Y"}.issubset(ready_set)
pe_family_ready = {"Y", "Z"}.issubset(ready_set)
full_ready = {"B", "Y", "Z"}.issubset(ready_set)
stage_ready = core_ready or pe_family_ready
analysis_ready = stage_ready

print(f"Ready cells:   {ready_cells if ready_cells else 'none'}")
print(f"Missing cells: {missing_cells if missing_cells else 'none'}")
print(f"Core B/Y ready: {'yes' if core_ready else 'no'}")
print(f"PE-family Y/Z ready: {'yes' if pe_family_ready else 'no'}")
print(f"Full B/Y/Z ready: {'yes' if full_ready else 'no'}")

if not core_ready and not pe_family_ready:
    print("\nScaffold mode: waiting on committed scenario JSONs for the first useful Experiment 2 comparison.")
elif pe_family_ready and not core_ready:
    print("\nPE-family follow-on mode: Y and Z are ready, so we can analyze PE vs Verified PE now while waiting on Cell B.")
elif core_ready and not full_ready:
    print("\nCore Experiment 2 mode: B and Y are ready. The notebook will write the honest AaT vs PE comparison and add Z later if it lands.")
else:
    print("\nFull Experiment 2 mode: B, Y, and Z are all ready.")


Ready cells:   ['B', 'Y']
Missing cells: ['Z']
Core B/Y ready: yes
PE-family Y/Z ready: no
Full B/Y/Z ready: no

Core Experiment 2 mode: B and Y are ready. The notebook will write the honest AaT vs PE comparison and add Z later if it lands.


## Per-scenario extraction

For each cell's latest run, extract per-scenario outcomes from the scenario JSONs. Failure detection per step follows the normalized-runner contract: one boolean per step, combining

- `step["success"] is False`, OR
- top-level `step["error"]` (truthy), OR
- `step["response"]["error"]` (truthy) — legacy path for pre-normalization artifacts

Counting each signal separately would double-count a single failed step once the runner normalizes tool errors at the step level. Judge scores live in `results/metrics/scenario_scores.jsonl` per `#17` — the loader normalizes `#113`/`#114` column names (`experiment_cell` → `cell`, `score_6d` → `judge_score`, `pass` → `judge_pass`) before the join.

In [6]:
def _step_failed(step: dict) -> bool:
    """One boolean per step under the normalized-runner contract.

    The runner sets `success=False` AND promotes `response["error"]` to `step["error"]`
    atomically when a tool call returns an error payload. So checking each signal
    separately would triple-count a single failed step. Here: OR them, count once.
    """
    if step.get("success") is False:
        return True
    if step.get("error"):
        return True
    resp = step.get("response")
    if isinstance(resp, dict) and resp.get("error"):
        return True
    return False

def _build_output_to_source_map(latest_run: Path) -> dict[Path, dict]:
    """Map resolved output_path -> {scenario_file (source), trial_index} via latencies.jsonl.

    Output JSON filenames embed cell/date/orchestration prefixes, so the same
    source scenario produces different output filenames across cells (e.g.
    `2026-04-26_B_aat_..._run01.json` vs `2026-04-13_Y_pe_..._run01.json`).
    Intersecting on output filename therefore never finds matches across cells.
    Use the source scenario_file recorded in latencies.jsonl as the join key
    instead.
    """
    rows, _ = _load_jsonl(latest_run / "latencies.jsonl")
    mapping: dict[Path, dict] = {}
    for row in rows:
        raw_path = row.get("output_path")
        if not raw_path:
            continue
        candidate = Path(raw_path)
        output_path = candidate if candidate.is_absolute() else REPO_ROOT / candidate
        try:
            resolved = output_path.resolve()
        except OSError:
            resolved = output_path
        mapping[resolved] = {
            "scenario_file": row.get("scenario_file"),
            "trial_index": row.get("trial_index"),
        }
    return mapping


def load_scenario_records(cell_spec: dict) -> pd.DataFrame:
    cell_dir = cell_spec["path"]
    raw_dir = cell_dir / "raw"
    latest_run = _latest_run_dir(raw_dir)
    if latest_run is None:
        return pd.DataFrame()
    output_to_source = _build_output_to_source_map(latest_run)
    rows = []
    parse_errors = []
    for scenario_file in _scenario_jsons(latest_run):
        try:
            data = json.loads(scenario_file.read_text())
        except json.JSONDecodeError as exc:
            parse_errors.append({"path": str(scenario_file), "reason": str(exc)})
            continue
        history = data.get("history") or []
        tool_errors = sum(1 for step in history if isinstance(step, dict) and _step_failed(step))

        scenario_obj = data.get("scenario")
        scenario_id = None
        canonical_id_present = False
        if isinstance(scenario_obj, dict):
            scenario_id = scenario_obj.get("id")
            canonical_id_present = scenario_id is not None
        if scenario_id is None:
            scenario_id = scenario_file.stem

        # Preserve legacy-null success as NaN so downstream aggregation doesn't
        # silently treat "unknown" as "failed".
        raw_success = data.get("success")
        success_value = raw_success if isinstance(raw_success, bool) else np.nan
        scenario_status = "canonical" if (canonical_id_present and isinstance(raw_success, bool)) else "legacy"

        # Prefer (source scenario_file, trial_index) from latencies.jsonl as the
        # cross-cell join key. Output JSON filenames embed cell/date prefixes
        # so they never match across cells (caught in PR #136 v2 review).
        try:
            resolved = scenario_file.resolve()
        except OSError:
            resolved = scenario_file
        latency_meta = output_to_source.get(resolved, {})
        source_scenario_file = latency_meta.get("scenario_file")
        latency_trial_index = latency_meta.get("trial_index")

        # Filename-derived trial_index is the legacy fallback for runs that
        # never wrote latencies.jsonl rows for this output. (Pre-PR #136 v3
        # version of this regex literal contained `\\d+` which matched the
        # literal characters; corrected to `\d+` here so the fallback works.)
        trial_match = re.search(r"_run(\d+)", scenario_file.stem)
        filename_trial_index = int(trial_match.group(1)) if trial_match else 1
        trial_index = latency_trial_index if latency_trial_index is not None else filename_trial_index

        rows.append({
            "cell": cell_spec["cell"],
            "label": cell_spec["label"],
            "run_id": latest_run.name,
            "scenario_id": scenario_id,
            # Source path from latencies.jsonl when available (preferred).
            "scenario_file": source_scenario_file or scenario_file.name,
            # Output-side artifact name preserved for debugging / artifact lookup.
            "output_file": scenario_file.name,
            "trial_index": trial_index,
            "scenario_status": scenario_status,
            "success": success_value,
            "failed_steps_count": len(data.get("failed_steps") or []),
            "history_length": len(history),
            "tool_error_count": tool_errors,
            "answer_length_chars": len(data.get("answer") or ""),
            "replans_used": (data.get("verification") or {}).get("replans_used") if isinstance(data.get("verification"), dict) else None,
        })
    if parse_errors:
        print(
            f"WARNING: {cell_spec['cell']}: {len(parse_errors)} unparseable scenario JSON file(s) in "
            f"{latest_run.name}; first: {parse_errors[0]['path']} ({parse_errors[0]['reason']})"
        )
    return pd.DataFrame(rows)

def load_judge_scores() -> pd.DataFrame:
    """Load judge scores from scenario_scores.jsonl, normalizing #113/#114 column names.

    Akshat's judge writes `experiment_cell`, `score_6d`, and `pass`. We coalesce
    to `cell` / `judge_score` / `judge_pass` so the downstream aggregation joins
    and aggregates on consistent names. Coalesce (not rename) handles the mixed-
    schema edge case where a JSONL accumulates both pre- and post-normalization
    rows: old rows fill NaN in the new columns without dropping anything.
    """
    scores_path = RESULTS_METRICS_DIR / "scenario_scores.jsonl"
    if not scores_path.exists():
        return pd.DataFrame()
    rows, parse_errors = _load_jsonl(scores_path)
    if parse_errors:
        print(
            f"WARNING: {len(parse_errors)} unparseable JSONL row(s) in {scores_path.name}; "
            f"first at line {parse_errors[0]['line_number']}: {parse_errors[0]['reason']}"
        )
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    # Coalesce old → new column names, preserving rows that already have the new name.
    coalesce_pairs = [
        ("cell", "experiment_cell"),
        ("judge_score", "score_6d"),
        ("judge_pass", "pass"),
    ]
    for new_col, old_col in coalesce_pairs:
        if new_col not in df.columns and old_col in df.columns:
            df[new_col] = df[old_col]
        elif new_col in df.columns and old_col in df.columns:
            df[new_col] = df[new_col].where(df[new_col].notna(), df[old_col])
    return df

if analysis_ready:
    scenario_df = pd.concat(
        [load_scenario_records(spec) for spec in CELL_SPECS],
        ignore_index=True,
    )
    judge_df = load_judge_scores()
    print(f"Scenario rows: {len(scenario_df)}")
    if not scenario_df.empty and "scenario_status" in scenario_df.columns:
        print("Scenario status counts:")
        print(scenario_df["scenario_status"].value_counts().to_string())
    print(f"Judge rows:    {len(judge_df)}")
    if not judge_df.empty:
        print(f"Judge cols:    {sorted(judge_df.columns)}")
    display(scenario_df.head(10))
else:
    scenario_df = pd.DataFrame()
    judge_df = pd.DataFrame()
    print("Skipping scenario extraction - cells missing captures.")

Scenario rows: 7
Scenario status counts:
scenario_status
legacy    7
Judge rows:    2
Judge cols:    ['cell', 'dim_agent_sequence_correct', 'dim_clarity_and_justification', 'dim_data_retrieval_accuracy', 'dim_generalized_result_verification', 'dim_hallucinations', 'dim_task_completion', 'experiment_cell', 'judge_model', 'judge_pass', 'judge_score', 'mcp_mode', 'model_id', 'orchestration_mode', 'pass', 'pass_threshold', 'run_name', 'scenario_file', 'scenario_id', 'schema_version', 'score_6d', 'scored_at', 'suggestions', 'trajectory_file', 'trial_index', 'wandb_run_url']


,cell,label,run_id,scenario_id,scenario_file,output_file,trial_index,scenario_status,success,failed_steps_count,history_length,tool_error_count,answer_length_chars,replans_used
0,B,AaT (MCP baseline),8979314_aat_mcp_baseline,2026-04-26_B_llama-3-1-8b-instruct_agent_as_to...,data/scenarios/multi_01_end_to_end_fault_respo...,2026-04-26_B_llama-3-1-8b-instruct_agent_as_to...,1,legacy,True,0,2,0,226,None
1,B,AaT (MCP baseline),8979314_aat_mcp_baseline,2026-04-26_B_llama-3-1-8b-instruct_agent_as_to...,data/scenarios/multi_01_end_to_end_fault_respo...,2026-04-26_B_llama-3-1-8b-instruct_agent_as_to...,2,legacy,True,0,2,0,202,None
2,B,AaT (MCP baseline),8979314_aat_mcp_baseline,2026-04-26_B_llama-3-1-8b-instruct_agent_as_to...,data/scenarios/multi_01_end_to_end_fault_respo...,2026-04-26_B_llama-3-1-8b-instruct_agent_as_to...,3,legacy,True,0,2,0,142,None
3,B,AaT (MCP baseline),8979314_aat_mcp_baseline,2026-04-26_B_llama-3-1-8b-instruct_agent_as_to...,data/scenarios/multi_02_dga_to_workorder_pipel...,2026-04-26_B_llama-3-1-8b-instruct_agent_as_to...,1,legacy,True,0,2,0,547,None
4,B,AaT (MCP baseline),8979314_aat_mcp_baseline,2026-04-26_B_llama-3-1-8b-instruct_agent_as_to...,data/scenarios/multi_02_dga_to_workorder_pipel...,2026-04-26_B_llama-3-1-8b-instruct_agent_as_to...,2,legacy,True,0,2,0,382,None
5,B,AaT (MCP baseline),8979314_aat_mcp_baseline,2026-04-26_B_llama-3-1-8b-instruct_agent_as_to...,data/scenarios/multi_02_dga_to_workorder_pipel...,2026-04-26_B_llama-3-1-8b-instruct_agent_as_to...,3,legacy,True,0,2,0,289,None
6,Y,Plan-Execute,local-20260413-003914_pe_mcp_baseline_watsonx_...,2026-04-13_Y_llama-3-3-70b-instruct_plan_execu...,data/scenarios/multi_01_end_to_end_fault_respo...,2026-04-13_Y_llama-3-3-70b-instruct_plan_execu...,1,legacy,NaN,0,8,0,424,None


## Comparison tables

This notebook can now emit two related aggregates:

- **Core Experiment 2 comparison** — B vs Y, with Z included when it is present
- **PE-family follow-on comparison** — Y vs Z, useful before AaT lands and also useful later as a focused PE-family ablation view

In both tables, aggregation is restricted to **canonical** scenario records (legacy pre-normalization artifacts are excluded because their `success` / `scenario.id` were null):

- **success rate** — fraction where `success=True`, over canonical scenarios only
- **mean failed steps** — average `len(failed_steps)` across canonical scenarios
- **mean history length** — average tool-call depth
- **mean tool-error count** — one count per failed step (see `_step_failed` above)
- **mean replans used** — Verified PE only; zero for AaT / plain PE

Judge pass rate joined when `results/metrics/scenario_scores.jsonl` exists, using the normalized columns from `load_judge_scores`.


In [7]:
def _cell_metadata(cell_id: str) -> dict:
    row = availability_df.loc[availability_df["cell"] == cell_id]
    if row.empty:
        return {}
    return row.iloc[0].to_dict()


def _check_metadata_compatibility(cell_ids: list[str]) -> list[str]:
    """Return a list of incompatibility reasons across the requested cells.

    Empty list = compatible. Each reason is a short human-readable string for
    the deferred-state note row and stdout warning. Compatibility means:
      - same scenario_set_name (human-readable guard)
      - same scenario_set_hash whenever hashes are present (correctness gate
        per the capture plan: same name with different contents would still
        produce an invalid join)
      - same model_id

    A mismatch is not a hard error; in preliminary mode we still publish a
    deferred CSV but refuse to aggregate, so downstream readers cannot mistake
    cross-set numbers for paired comparisons.
    """
    metas = {cell: _cell_metadata(cell) for cell in cell_ids}
    reasons = []

    scenario_sets = {cell: metas[cell].get("scenario_set_name") for cell in cell_ids}
    scenario_hashes = {cell: metas[cell].get("scenario_set_hash") for cell in cell_ids}
    model_ids = {cell: metas[cell].get("model_id") for cell in cell_ids}

    if len({v for v in scenario_sets.values() if v}) > 1:
        reasons.append(f"scenario_set_name mismatch across {cell_ids}: {scenario_sets}")

    # Hash check is the correctness gate. Compare only across cells that
    # actually carry a hash so a missing hash on legacy artifacts does not
    # spuriously trigger the gate; once any two cells both expose hashes,
    # they must match.
    cells_with_hash = {cell: h for cell, h in scenario_hashes.items() if h}
    if len({v for v in cells_with_hash.values()}) > 1:
        reasons.append(f"scenario_set_hash mismatch across {sorted(cells_with_hash)}: {cells_with_hash}")

    if len({v for v in model_ids.values() if v}) > 1:
        reasons.append(f"model_id mismatch across {cell_ids}: {model_ids}")

    return reasons


def _restrict_to_common_pairs(subset: pd.DataFrame, cell_ids: list[str]) -> pd.DataFrame:
    """Restrict `subset` to rows whose (scenario_file, trial_index) appears in EVERY cell_id.

    Required for any meaningful preliminary cross-cell aggregation: aggregating
    over the union (rather than the intersection) compares cells on different
    scenario sets, which is exactly the Critical issue the v1 review caught.
    """
    if "scenario_file" not in subset.columns or "trial_index" not in subset.columns:
        return subset.iloc[0:0]

    by_cell = {
        cell: set(map(tuple, group[["scenario_file", "trial_index"]].itertuples(index=False, name=None)))
        for cell, group in subset.groupby("cell")
    }
    if not by_cell:
        return subset.iloc[0:0]
    common = set.intersection(*[by_cell.get(cell, set()) for cell in cell_ids])
    if not common:
        return subset.iloc[0:0]

    keys = pd.MultiIndex.from_tuples(sorted(common), names=["scenario_file", "trial_index"])
    indexed = subset.set_index(["scenario_file", "trial_index"])
    return indexed.loc[indexed.index.intersection(keys)].reset_index()


def aggregate_cells(cell_ids: list[str], output_path: Path, label: str) -> pd.DataFrame:
    base_subset = scenario_df[scenario_df["cell"].isin(cell_ids)].copy()
    if base_subset.empty:
        _write_deferred_csv(
            output_path, ORCHESTRATION_CSV_COLUMNS,
            f"{label} deferred - no scenario rows for cells {cell_ids}.",
        )
        print(f"{label} deferred - wrote schema-only {output_path.relative_to(REPO_ROOT)}.")
        return pd.DataFrame()

    canonical = base_subset[base_subset["scenario_status"] == "canonical"].copy()
    incompat_reasons = _check_metadata_compatibility(cell_ids)

    # Metadata incompatibility is a refusal condition for BOTH preliminary and
    # canonical aggregation. The capture plan requires comparisons to join on
    # (scenario_set_hash, model_id); even canonical scenario.id rows compared
    # across mismatched scenario sets / models / hashes silently publish an
    # invalid aggregate. Refuse before either branch builds an aggregate.
    if incompat_reasons:
        note = (
            f"{label} deferred (refused): incompatible metadata - "
            + "; ".join(incompat_reasons)
        )
        _write_deferred_csv(output_path, ORCHESTRATION_CSV_COLUMNS, note)
        print(note)
        print(f"Wrote schema-only {output_path.relative_to(REPO_ROOT)}.")
        return pd.DataFrame()

    if canonical.empty:
        legacy = base_subset.copy()
        if legacy["success"].isna().all():
            note = (
                f"{label} deferred - cells {cell_ids} have data but no canonical "
                f"scenario.id propagation AND no legacy success signal; nothing to aggregate."
            )
            _write_deferred_csv(output_path, ORCHESTRATION_CSV_COLUMNS, note)
            print(note)
            return pd.DataFrame()

        common = _restrict_to_common_pairs(legacy, cell_ids)
        if common.empty:
            note = (
                f"{label} deferred (PRELIMINARY refused): no shared (scenario_file, trial_index) "
                f"keys across cells {cell_ids}. Aggregating over the union would compare different "
                f"scenario sets - rejecting silently invalid cross-run comparison."
            )
            _write_deferred_csv(output_path, ORCHESTRATION_CSV_COLUMNS, note)
            print(note)
            return pd.DataFrame()

        coverage_msg = (
            f"{label} running in PRELIMINARY mode on {len(common)} shared (scenario_file, trial_index) "
            f"pair(s) across cells {cell_ids}. Cells lack canonical scenario.id propagation in run output "
            f"JSONs (upstream AaT runner gap). Numbers are publishable as preliminary estimates only."
        )
        print(coverage_msg)
        subset = common
        mode = "preliminary"
    else:
        legacy_dropped = len(base_subset) - len(canonical)
        if legacy_dropped:
            print(
                f"Dropping {legacy_dropped} legacy scenario record(s) from {label.lower()} "
                f"(null success or missing scenario.id)."
            )
        canonical_common = _restrict_to_common_pairs(canonical, cell_ids)
        if canonical_common.empty:
            note = (
                f"{label} deferred - no shared (scenario_file, trial_index) keys across canonical "
                f"runs of cells {cell_ids}; cannot form paired comparison."
            )
            _write_deferred_csv(output_path, ORCHESTRATION_CSV_COLUMNS, note)
            print(note)
            return pd.DataFrame()
        subset = canonical_common
        mode = "canonical"

    agg = (
        subset.groupby(["cell", "label"], as_index=False)
        .agg(
            scenarios=("scenario_id", "count"),
            success_rate=("success", "mean"),
            mean_failed_steps=("failed_steps_count", "mean"),
            mean_history_length=("history_length", "mean"),
            mean_tool_errors=("tool_error_count", "mean"),
            mean_replans=("replans_used", "mean"),
        )
    )
    agg.insert(2, "mode", mode)

    if not judge_df.empty and {"cell", "scenario_id"}.issubset(judge_df.columns):
        subset_cols = ["cell", "scenario_id", "trial_index"]
        if "run_id" in subset.columns:
            subset_cols.append("run_id")
        subset_keys = subset[subset_cols].drop_duplicates().copy()

        join_keys = ["cell", "scenario_id", "trial_index"]
        if "run_name" in judge_df.columns and "run_id" in subset_keys.columns:
            subset_keys = subset_keys.rename(columns={"run_id": "run_name"})
            join_keys = ["cell", "scenario_id", "trial_index", "run_name"]

        merge_on = [k for k in join_keys if k in subset_keys.columns and k in judge_df.columns]
        if merge_on:
            judge_matched = judge_df.merge(
                subset_keys[merge_on].drop_duplicates(),
                on=merge_on,
                how="inner",
            )
            dedup_keys = merge_on
            if "scored_at" in judge_matched.columns:
                judge_matched = (
                    judge_matched
                    .sort_values("scored_at")
                    .drop_duplicates(subset=dedup_keys, keep="last")
                )
            else:
                judge_matched = judge_matched.drop_duplicates(subset=dedup_keys, keep="last")

            if not judge_matched.empty:
                agg_kwargs = {"judge_scenarios": ("scenario_id", "count")}
                if "judge_score" in judge_matched.columns:
                    agg_kwargs["judge_score_mean"] = ("judge_score", "mean")
                if "judge_pass" in judge_matched.columns:
                    agg_kwargs["judge_pass_rate"] = ("judge_pass", "mean")
                judge_agg = judge_matched.groupby("cell", as_index=False).agg(**agg_kwargs)
                agg = agg.merge(judge_agg, on="cell", how="left")

    agg.to_csv(output_path, index=False)
    print(f"Wrote {output_path.relative_to(REPO_ROOT)} ({mode} mode)")
    display(agg)
    return agg

pe_family_agg = pd.DataFrame()
if pe_family_ready and not scenario_df.empty:
    pe_family_agg = aggregate_cells(
        ["Y", "Z"],
        PE_FAMILY_CSV,
        "PE-family follow-on comparison",
    )
else:
    _write_deferred_csv(
        PE_FAMILY_CSV, ORCHESTRATION_CSV_COLUMNS,
        "PE-family follow-on deferred - need both Y and Z scenario captures.",
    )
    print(f"PE-family follow-on comparison deferred - wrote schema-only {PE_FAMILY_CSV.relative_to(REPO_ROOT)}.")

orchestration_agg = pd.DataFrame()
core_cells = ["B", "Y"] + (["Z"] if "Z" in ready_set else [])
if core_ready and not scenario_df.empty:
    orchestration_agg = aggregate_cells(
        core_cells,
        ORCHESTRATION_CSV,
        "Core Experiment 2 comparison",
    )
else:
    _write_deferred_csv(
        ORCHESTRATION_CSV, ORCHESTRATION_CSV_COLUMNS,
        "Core orchestration comparison deferred - need at least Cells B and Y.",
    )
    print(f"Core orchestration comparison deferred - wrote schema-only {ORCHESTRATION_CSV.relative_to(REPO_ROOT)}.")

PE-family follow-on comparison deferred - wrote schema-only results/metrics/notebook03_pe_family_follow_on.csv.
Core Experiment 2 comparison deferred (refused): incompatible metadata - scenario_set_name mismatch across ['B', 'Y']: {'B': 'smartgrid_multi_domain', 'Y': 'smartgrid_multi_domain_smoke'}
Wrote schema-only results/metrics/notebook03_orchestration_comparison.csv.


## Self-Ask ablations

`#24` shipped the Self-Ask hook for the repo-local PE-family runners. The ablation pass here is **run-centric**, not cell-centric: Y and Z reuse the same benchmark directories over time, so we scan the raw run inventory, classify each run as baseline vs Self-Ask from the committed artifacts, and then compare the latest ready pair per orchestration lane when both variants exist.


In [8]:
SELF_ASK_ABLATION_SPECS = [
    {
        "cell": "Y",
        "comparison_label": "Plan-Execute Self-Ask ablation",
        "baseline_label": "Plan-Execute",
        "self_ask_label": "Plan-Execute + Self-Ask",
        "path": BENCHMARKS_DIR / "cell_Y_plan_execute",
    },
    {
        "cell": "Z",
        "comparison_label": "Verified PE Self-Ask ablation",
        "baseline_label": "Verified PE",
        "self_ask_label": "Verified PE + Self-Ask",
        "path": BENCHMARKS_DIR / "cell_Z_hybrid",
    },
]

SELF_ASK_VARIANT_ORDER = {"baseline": 0, "self_ask": 1}
SELF_ASK_VARIANT_COLORS = {"baseline": "#4c78a8", "self_ask": "#72b7b2"}
SELF_ASK_INVENTORY_COLUMNS = [
    "cell",
    "comparison_label",
    "variant_key",
    "variant_label",
    "signal_source",
    "run_id",
    "run_name",
    "benchmark_config_path",
    "started_at",
    "latency_rows",
    "scenario_json_count",
    "canonical_rows",
    "run_ready",
    "success_rate",
    "latency_p50",
    "latency_p95",
]


def _run_meta(run_dir: Path) -> dict:
    return _load_json(run_dir / "meta.json") or {}


def _run_sort_value(run_dir: Path, meta: dict | None = None) -> float:
    meta = meta or _run_meta(run_dir)
    parsed = _parse_iso_ts(meta.get("started_at") or meta.get("finished_at"))
    if parsed is not None:
        try:
            return parsed.timestamp()
        except (OSError, OverflowError, ValueError):
            pass
    return run_dir.stat().st_mtime


def _classify_self_ask_variant(run_dir: Path) -> tuple[str, str, dict]:
    meta = _run_meta(run_dir)
    name_hints = any(
        "self_ask" in str(value).lower()
        for value in (run_dir.name, meta.get("run_name", ""), meta.get("benchmark_config_path", ""))
    )

    payload_signal = None
    for scenario_file in _scenario_jsons(run_dir):
        payload = _load_json(scenario_file) or {}
        if "self_ask" not in payload:
            continue
        self_ask = payload.get("self_ask")
        if isinstance(self_ask, dict):
            if "enabled" in self_ask:
                payload_signal = bool(self_ask["enabled"])
            else:
                payload_signal = True
        else:
            payload_signal = bool(self_ask)
        break

    if payload_signal is not None:
        return ("self_ask" if payload_signal else "baseline"), "payload", meta
    if name_hints:
        return "self_ask", "name_hint", meta
    return "baseline", "default_baseline", meta


def load_run_scenario_records(cell_spec: dict, run_dir: Path, variant_key: str, variant_label: str, run_name: str) -> pd.DataFrame:
    output_to_source = _build_output_to_source_map(run_dir)
    rows = []
    parse_errors = []
    for scenario_file in _scenario_jsons(run_dir):
        try:
            data = json.loads(scenario_file.read_text())
        except json.JSONDecodeError as exc:
            parse_errors.append({"path": str(scenario_file), "reason": str(exc)})
            continue
        history = data.get("history") or data.get("trajectory") or []
        tool_errors = sum(1 for step in history if isinstance(step, dict) and _step_failed(step))

        scenario_obj = data.get("scenario")
        scenario_id = None
        canonical_id_present = False
        if isinstance(scenario_obj, dict):
            scenario_id = scenario_obj.get("id")
            canonical_id_present = scenario_id is not None
        if scenario_id is None:
            scenario_id = scenario_file.stem

        raw_success = data.get("success")
        success_value = raw_success if isinstance(raw_success, bool) else np.nan
        scenario_status = "canonical" if (canonical_id_present and isinstance(raw_success, bool)) else "legacy"

        try:
            resolved = scenario_file.resolve()
        except OSError:
            resolved = scenario_file
        latency_meta = output_to_source.get(resolved, {})
        source_scenario_file = latency_meta.get("scenario_file")
        latency_trial_index = latency_meta.get("trial_index")

        trial_match = re.search(r"_run(\d+)", scenario_file.stem)
        filename_trial_index = int(trial_match.group(1)) if trial_match else 1
        trial_index = latency_trial_index if latency_trial_index is not None else filename_trial_index

        replans_used = None
        verification = data.get("verification")
        if isinstance(verification, dict):
            replans_used = verification.get("replans_used")

        rows.append({
            "cell": cell_spec["cell"],
            "comparison_label": cell_spec["comparison_label"],
            "variant_key": variant_key,
            "variant_label": variant_label,
            "run_id": run_dir.name,
            "run_name": run_name,
            "scenario_id": scenario_id,
            "scenario_file": source_scenario_file or scenario_file.name,
            "output_file": scenario_file.name,
            "trial_index": trial_index,
            "scenario_status": scenario_status,
            "success": success_value,
            "failed_steps_count": len(data.get("failed_steps") or []),
            "history_length": len(history),
            "tool_error_count": tool_errors,
            "answer_length_chars": len(data.get("answer") or ""),
            "replans_used": replans_used,
        })
    if parse_errors:
        print(
            f"WARNING: self-ask scan {cell_spec['cell']} {run_dir.name}: {len(parse_errors)} unparseable "
            f"scenario JSON file(s); first: {parse_errors[0]['path']}"
        )
    return pd.DataFrame(rows)


def scan_self_ask_ablation_runs(cell_spec: dict) -> tuple[pd.DataFrame, dict]:
    raw_dir = cell_spec["path"] / "raw"
    run_rows = []
    records_by_run = {}
    if not raw_dir.exists():
        return pd.DataFrame(columns=SELF_ASK_INVENTORY_COLUMNS), records_by_run

    for run_dir in sorted([p for p in raw_dir.iterdir() if p.is_dir()]):
        variant_key, signal_source, meta = _classify_self_ask_variant(run_dir)
        variant_label = cell_spec["self_ask_label"] if variant_key == "self_ask" else cell_spec["baseline_label"]
        run_name = meta.get("run_name") or run_dir.name
        records = load_run_scenario_records(cell_spec, run_dir, variant_key, variant_label, run_name)
        records_by_run[run_dir.name] = records

        canonical = records[records["scenario_status"] == "canonical"].copy() if not records.empty else pd.DataFrame()
        latencies, latency_parse_errors = _load_jsonl(run_dir / "latencies.jsonl")
        if latency_parse_errors:
            print(
                f"WARNING: {cell_spec['cell']} run {run_dir.name}: {len(latency_parse_errors)} unparseable "
                f"latencies.jsonl row(s); first at line {latency_parse_errors[0]['line_number']}"
            )
        latency_values = [row.get("latency_seconds") for row in latencies if isinstance(row.get("latency_seconds"), (int, float))]

        run_rows.append({
            "cell": cell_spec["cell"],
            "comparison_label": cell_spec["comparison_label"],
            "variant_key": variant_key,
            "variant_label": variant_label,
            "signal_source": signal_source,
            "run_id": run_dir.name,
            "run_name": run_name,
            "benchmark_config_path": meta.get("benchmark_config_path"),
            "started_at": meta.get("started_at") or meta.get("finished_at") or "",
            "sort_value": _run_sort_value(run_dir, meta),
            "latency_rows": len(latencies),
            "scenario_json_count": len(_scenario_jsons(run_dir)),
            "canonical_rows": len(canonical),
            "run_ready": bool(len(latencies) > 0 and len(canonical) > 0),
            "success_rate": canonical["success"].mean() if not canonical.empty else np.nan,
            "latency_p50": pd.Series(latency_values).quantile(0.50) if latency_values else np.nan,
            "latency_p95": pd.Series(latency_values).quantile(0.95) if latency_values else np.nan,
        })

    inventory_df = pd.DataFrame(run_rows)
    if inventory_df.empty:
        inventory_df = pd.DataFrame(columns=SELF_ASK_INVENTORY_COLUMNS + ["sort_value"])
    return inventory_df, records_by_run


def aggregate_self_ask_ablation(records_df: pd.DataFrame) -> pd.DataFrame:
    subset = records_df[records_df["scenario_status"] == "canonical"].copy()
    if subset.empty:
        return pd.DataFrame()

    agg = (
        subset.groupby(
            ["cell", "comparison_label", "variant_key", "variant_label", "run_id", "run_name"],
            as_index=False,
        )
        .agg(
            scenarios=("scenario_id", "count"),
            success_rate=("success", "mean"),
            mean_failed_steps=("failed_steps_count", "mean"),
            mean_history_length=("history_length", "mean"),
            mean_tool_errors=("tool_error_count", "mean"),
            mean_replans=("replans_used", "mean"),
        )
    )

    if not judge_df.empty and {"cell", "scenario_id", "trial_index"}.issubset(judge_df.columns):
        subset_keys = subset[["cell", "scenario_id", "trial_index", "run_name"]].drop_duplicates().copy()
        merge_on = [key for key in ["cell", "scenario_id", "trial_index", "run_name"] if key in judge_df.columns]
        if merge_on:
            judge_matched = judge_df.merge(subset_keys[merge_on].drop_duplicates(), on=merge_on, how="inner")
            if not judge_matched.empty:
                if "scored_at" in judge_matched.columns:
                    judge_matched = (
                        judge_matched.sort_values("scored_at")
                        .drop_duplicates(subset=merge_on, keep="last")
                    )
                else:
                    judge_matched = judge_matched.drop_duplicates(subset=merge_on, keep="last")

                judge_group_keys = ["cell", "run_name"]
                judge_kwargs = {"judge_scenarios": ("scenario_id", "count")}
                if "judge_score" in judge_matched.columns:
                    judge_kwargs["judge_score_mean"] = ("judge_score", "mean")
                if "judge_pass" in judge_matched.columns:
                    judge_kwargs["judge_pass_rate"] = ("judge_pass", "mean")
                judge_agg = judge_matched.groupby(judge_group_keys, as_index=False).agg(**judge_kwargs)
                agg = agg.merge(judge_agg, on=["cell", "run_name"], how="left")

    return agg.sort_values(["cell", "variant_key"], key=lambda s: s.map(SELF_ASK_VARIANT_ORDER).fillna(99) if s.name == "variant_key" else s)


def plot_self_ask_ablation(summary_df: pd.DataFrame):
    cells = summary_df["cell"].drop_duplicates().tolist()
    fig, axes = plt.subplots(len(cells), 2, figsize=(11, 4.5 * len(cells)), squeeze=False)

    for row_idx, cell in enumerate(cells):
        cell_df = summary_df[summary_df["cell"] == cell].copy()
        cell_df["variant_order"] = cell_df["variant_key"].map(SELF_ASK_VARIANT_ORDER)
        cell_df = cell_df.sort_values("variant_order")
        labels = cell_df["variant_label"].tolist()
        colors = [SELF_ASK_VARIANT_COLORS[key] for key in cell_df["variant_key"]]

        success_ax = axes[row_idx][0]
        success_ax.bar(labels, cell_df["success_rate"], color=colors, edgecolor="black", linewidth=0.5)
        success_ax.set_ylim(0, 1.05)
        success_ax.set_title(f"{cell_df['comparison_label'].iloc[0]} - success rate")
        success_ax.set_ylabel("Fraction of canonical scenarios with success=True")
        success_ax.grid(axis="y", alpha=0.3)

        latency_ax = axes[row_idx][1]
        p50 = cell_df["latency_p50"].astype(float).to_numpy()
        p95 = cell_df["latency_p95"].astype(float).to_numpy()
        bars = latency_ax.bar(labels, p50, color=colors, edgecolor="black", linewidth=0.5)
        for bar, mid, high in zip(bars, p50, p95):
            if np.isnan(mid) or np.isnan(high):
                continue
            cx = bar.get_x() + bar.get_width() / 2
            latency_ax.plot([cx, cx], [mid, high], color="black", linewidth=1)
            latency_ax.plot([cx - 0.12, cx + 0.12], [high, high], color="black", linewidth=1)
        latency_ax.set_title(f"{cell_df['comparison_label'].iloc[0]} - latency (p50 bar, p95 cap)")
        latency_ax.set_ylabel("End-to-end latency (seconds)")
        latency_ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(SELF_ASK_FIGURE_PNG, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Wrote {SELF_ASK_FIGURE_PNG.relative_to(REPO_ROOT)}")


self_ask_inventory_parts = []
self_ask_records = []
for spec in SELF_ASK_ABLATION_SPECS:
    inventory_part, records_by_run = scan_self_ask_ablation_runs(spec)
    if not inventory_part.empty:
        self_ask_inventory_parts.append(inventory_part)
    self_ask_records.extend(records_by_run.values())

if self_ask_inventory_parts:
    self_ask_inventory_df = pd.concat(self_ask_inventory_parts, ignore_index=True)
else:
    self_ask_inventory_df = pd.DataFrame(columns=SELF_ASK_INVENTORY_COLUMNS + ["sort_value"])

self_ask_inventory_df.drop(columns=["sort_value"], errors="ignore").to_csv(SELF_ASK_INVENTORY_CSV, index=False)
print(f"Wrote {SELF_ASK_INVENTORY_CSV.relative_to(REPO_ROOT)}")
if not self_ask_inventory_df.empty:
    display(self_ask_inventory_df.drop(columns=["sort_value"], errors="ignore"))
else:
    print("No Y/Z raw runs found for the Self-Ask ablation inventory yet.")

selected_self_ask_runs = pd.DataFrame()
if not self_ask_inventory_df.empty:
    ready_runs = self_ask_inventory_df[self_ask_inventory_df["run_ready"]].copy()
    if not ready_runs.empty:
        selected_self_ask_runs = (
            ready_runs.sort_values(["cell", "variant_key", "sort_value"])
            .groupby(["cell", "variant_key"], as_index=False)
            .tail(1)
            .sort_values(["cell", "variant_key"], key=lambda s: s.map(SELF_ASK_VARIANT_ORDER).fillna(99) if s.name == "variant_key" else s)
        )

self_ask_ablation_df = pd.DataFrame()
selected_self_ask_records = []
if not selected_self_ask_runs.empty:
    available_cells = []
    for cell, group in selected_self_ask_runs.groupby("cell"):
        if {"baseline", "self_ask"}.issubset(set(group["variant_key"])):
            available_cells.append(cell)
    if available_cells:
        wanted_ids = set(selected_self_ask_runs[selected_self_ask_runs["cell"].isin(available_cells)]["run_id"])
        selected_self_ask_records = [df for df in self_ask_records if not df.empty and df["run_id"].iloc[0] in wanted_ids]
        if selected_self_ask_records:
            selected_records_df = pd.concat(selected_self_ask_records, ignore_index=True)
            self_ask_ablation_df = aggregate_self_ask_ablation(selected_records_df)

if not self_ask_ablation_df.empty:
    self_ask_ablation_df.to_csv(SELF_ASK_ABLATION_CSV, index=False)
    print(f"Wrote {SELF_ASK_ABLATION_CSV.relative_to(REPO_ROOT)}")
    display(self_ask_ablation_df)
    plot_self_ask_ablation(self_ask_ablation_df)
else:
    _write_deferred_csv(
        SELF_ASK_ABLATION_CSV, SELF_ASK_ABLATION_COLUMNS,
        "Self-Ask ablation deferred - need a ready baseline/self-ask run pair for Y and/or Z.",
    )
    _write_deferred_figure(
        SELF_ASK_FIGURE_PNG,
        "Self-Ask ablation - deferred",
        "Pending: at least one ready (baseline, self_ask) run pair\nfor Cell Y and/or Cell Z.",
    )
    print(f"Self-Ask ablation deferred - wrote schema-only {SELF_ASK_ABLATION_CSV.relative_to(REPO_ROOT)} "
          f"and placeholder {SELF_ASK_FIGURE_PNG.relative_to(REPO_ROOT)}.")


Wrote results/metrics/notebook03_self_ask_run_inventory.preflight.csv


,cell,comparison_label,variant_key,variant_label,signal_source,run_id,run_name,benchmark_config_path,started_at,latency_rows,scenario_json_count,canonical_rows,run_ready,success_rate,latency_p50,latency_p95
0,Y,Plan-Execute Self-Ask ablation,baseline,Plan-Execute,default_baseline,issue18-smartgrid-smoke,issue18-smartgrid-smoke,NaN,,0,1,0,False,NaN,NaN,NaN
1,Y,Plan-Execute Self-Ask ablation,baseline,Plan-Execute,default_baseline,issue3-aob-harness-smoke,issue3-aob-harness-smoke,NaN,,0,1,0,False,NaN,NaN,NaN
2,Y,Plan-Execute Self-Ask ablation,baseline,Plan-Execute,default_baseline,local-20260413-003914_pe_mcp_baseline_watsonx_...,local-20260413-003914_pe_mcp_baseline_watsonx_...,/tmp/smartgrid_watsonx_smoke.XXXXXX.env,2026-04-13T04:39:14.266300+00:00,1,1,0,False,NaN,93.64177,93.64177


Self-Ask ablation deferred - wrote schema-only results/metrics/notebook03_self_ask_ablation.csv and placeholder results/figures/notebook03_self_ask_ablation.png.


## Figures — staged experiment outputs

The core Experiment 2 figure remains the paper-facing success-rate + latency comparison.

When Y and Z are available before B, the notebook should also emit a smaller PE-family follow-on figure so we can inspect the runnable PE/Verified-PE lane without pretending the AaT baseline already exists.


In [9]:
COLOR_MAP = {
    "B": "#f58518",  # keep Cell B aligned with Notebook 02's MCP-baseline orange
    "Y": "#4c78a8",
    "Z": "#e45756",
}


def plot_comparison(agg_df: pd.DataFrame, cell_ids: list[str], figure_path: Path, title_prefix: str):
    """Render the standard 2-panel (success-rate, latency) comparison.

    Reads success_rate from the already-computed `agg_df` so canonical AND
    preliminary modes both populate the panel. Latency p50/p95 still come from
    `availability_df` (latencies.jsonl is the source for cell-level latency
    summaries; agg_df aggregates per-scenario success only).

    The previous version recomputed success_rate from
    `scenario_df[scenario_status == "canonical"]`, which left the success-rate
    panel blank whenever the CSV was in preliminary mode. v2 fix.
    """
    summary = availability_df.set_index("cell").loc[cell_ids]
    p50 = summary["latency_seconds_p50"].astype(float).values
    p95 = summary["latency_seconds_p95"].astype(float).values
    labels = summary["label"].tolist()
    colors = [COLOR_MAP[cell] for cell in cell_ids]

    success_indexed = (
        agg_df.set_index("cell").reindex(cell_ids)["success_rate"].astype(float).values
        if not agg_df.empty
        else np.array([np.nan] * len(cell_ids))
    )
    mode_label = ""
    if not agg_df.empty and "mode" in agg_df.columns:
        modes = sorted(set(agg_df["mode"].dropna()))
        if modes:
            mode_label = f" ({'/'.join(modes)} mode)"

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

    sr_bars = ax1.bar(labels, np.nan_to_num(success_indexed, nan=0.0), color=colors,
                      edgecolor="black", linewidth=0.5)
    for bar, value in zip(sr_bars, success_indexed):
        if np.isnan(value):
            bar.set_hatch("//")
            bar.set_facecolor("none")
            bar.set_edgecolor("#666666")
            cx = bar.get_x() + bar.get_width() / 2
            ax1.text(cx, 0.05, "no data", ha="center", va="bottom", fontsize=9, color="#444444")
    ax1.set_ylim(0, 1.05)
    ax1.set_title(f"{title_prefix} - success rate{mode_label}")
    ax1.set_ylabel("Fraction of scenarios with success=True")
    ax1.grid(axis="y", alpha=0.3)

    bars = ax2.bar(labels, p50, color=colors, edgecolor="black", linewidth=0.5)
    for x, mid, high in zip(bars, p50, p95):
        if np.isnan(mid) or np.isnan(high):
            continue
        cx = x.get_x() + x.get_width() / 2
        ax2.plot([cx, cx], [mid, high], color="black", linewidth=1)
        ax2.plot([cx - 0.12, cx + 0.12], [high, high], color="black", linewidth=1)
    ax2.set_title(f"{title_prefix} - latency (p50 bar, p95 cap)")
    ax2.set_ylabel("End-to-end latency (seconds)")
    ax2.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(figure_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Wrote {figure_path.relative_to(REPO_ROOT)}{mode_label}")


if core_ready and not orchestration_agg.empty and not availability_df.empty:
    core_figure_cells = ["B", "Y"] + (["Z"] if "Z" in ready_set else [])
    plot_comparison(
        orchestration_agg,
        core_figure_cells,
        ORCHESTRATION_FIGURE_PNG,
        "Experiment 2 core comparison",
    )
else:
    _write_deferred_figure(
        ORCHESTRATION_FIGURE_PNG,
        "Experiment 2 core comparison - deferred",
        "Pending: at least Cells B and Y\n(canonical or preliminary scenario captures with shared join keys).",
    )
    print(f"Core figure deferred - wrote placeholder {ORCHESTRATION_FIGURE_PNG.relative_to(REPO_ROOT)}.")

if pe_family_ready and not pe_family_agg.empty and not availability_df.empty:
    plot_comparison(
        pe_family_agg,
        ["Y", "Z"],
        PE_FAMILY_FIGURE_PNG,
        "PE-family follow-on",
    )
else:
    _write_deferred_figure(
        PE_FAMILY_FIGURE_PNG,
        "PE-family follow-on - deferred",
        "Pending: both Cell Y and Cell Z scenario captures\nwith shared (scenario_file, trial_index) keys.",
    )
    print(f"PE-family follow-on figure deferred - wrote placeholder {PE_FAMILY_FIGURE_PNG.relative_to(REPO_ROOT)}.")

Core figure deferred - wrote placeholder results/figures/notebook03_orchestration_comparison.png.


PE-family follow-on figure deferred - wrote placeholder results/figures/notebook03_pe_family_follow_on.png.


## Failure pattern — where does each orchestration break?

When `failed_steps` or `tool_error_count` is non-zero, where in the trajectory does it happen? For PE / Verified PE, step index tells us whether failures cluster at plan formation vs execution vs verification. For AaT, step index corresponds to ReAct-loop iteration.

**Recovery rate semantics:** for a cell with zero failures, recovery_rate is **NaN** (not 0), because "no failures happened" is not the same signal as "failures happened and none recovered." Displayed separately via `any_failure` so the reader can distinguish the two.

In [10]:
if analysis_ready and not scenario_df.empty:
    canonical = scenario_df[scenario_df["scenario_status"] == "canonical"]
    if canonical.empty:
        _write_deferred_csv(
            FAILURE_BREAKDOWN_CSV, FAILURE_BREAKDOWN_COLUMNS,
            "Failure breakdown deferred - no canonical scenario rows yet.",
        )
        print(f"Failure breakdown deferred - wrote schema-only {FAILURE_BREAKDOWN_CSV.relative_to(REPO_ROOT)}.")
    else:
        failure_breakdown = (
            canonical.assign(
                any_failure=(canonical["failed_steps_count"] > 0) | (canonical["tool_error_count"] > 0),
                recovered=(canonical["failed_steps_count"] > 0) & (canonical["success"] == True),
            )
            .groupby(["cell", "label"], as_index=False)
            .agg(
                scenarios=("scenario_id", "count"),
                any_failure=("any_failure", "sum"),
                recovered=("recovered", "sum"),
            )
        )

        # NaN for zero-failure cells so "not applicable" is distinguishable from "0%
        # recovery". Do NOT .fillna(0).
        failure_breakdown["recovery_rate"] = (
            failure_breakdown["recovered"]
            / failure_breakdown["any_failure"].where(failure_breakdown["any_failure"] > 0)
        )

        failure_breakdown.to_csv(FAILURE_BREAKDOWN_CSV, index=False)
        print(f"Wrote {FAILURE_BREAKDOWN_CSV.relative_to(REPO_ROOT)}")
        display(failure_breakdown)
else:
    _write_deferred_csv(
        FAILURE_BREAKDOWN_CSV, FAILURE_BREAKDOWN_COLUMNS,
        "Failure breakdown deferred - need scenario captures.",
    )
    print(f"Failure breakdown deferred - wrote schema-only {FAILURE_BREAKDOWN_CSV.relative_to(REPO_ROOT)}.")

Failure breakdown deferred - wrote schema-only results/metrics/notebook03_failure_breakdown.csv.


## What changes later

Forward-looking deltas when real Experiment 2 captures land:

- per-step latency extraction from scenario JSONs (each `history` entry carries a latency if the runner records it)
- judge score join (`#17` — once `results/metrics/scenario_scores.jsonl` has entries from Maverick-17B)
- replan behavior distribution for Cell Z (histogram of `replans_used` — does verification actually help or just add cost?)
- AaT vs PE qualitative analysis — does PE need fewer tool calls because the plan is committed upfront?
- baseline Self-Ask ablations via `configs/experiment2/exp2_cell_Y_pe_self_ask_mcp_baseline.env` and `configs/experiment2/exp2_cell_Z_verified_pe_self_ask_mcp_baseline.env`
- only after the optimized transport is behaviorally real outside AaT: consider the planned `Y/Z + Self-Ask + MCP optimized` follow-ons

**Dependencies worth tracking:**

- `#104` / `#25` AaT runner work — makes Cell B real and upgrades this notebook from PE-family follow-on mode to the honest core Experiment 2 comparison
- `#17` judge wiring (Akshat) — feeds judge score join (PR `#113` merged; live value now depends on refreshed judge artifacts)
- `#23` / `#24` merges — already provide the Cell Z / Self-Ask method paths; what remains is scenario-scale capture, not runner plumbing
- `#115` server hardening merge — stabilizes Cell Y's tool-error baseline (tool-error count is load-bearing for the "recovery" analysis above)
